# Tema 8 · Laboratorio — Overfitting y regularización

**Aprendizaje Profundo · CUNEF Universidad**

En este laboratorio **provocamos overfitting** a propósito y luego lo **arreglamos** con las herramientas del tema: dropout, regularización L2 y early stopping.

La receta para forzar el sobreajuste: **pocos datos** + **modelo grande** + **muchas épocas**.

> Ejecuta las celdas en orden. En Colab no necesitas instalar nada.

## 1 · Datos: Fashion-MNIST, pero pocos

Nos quedamos con **solo 1 500** imágenes de entrenamiento. Con tan pocos datos y una red grande, el overfitting aparece enseguida.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Pocos datos de entrenamiento -> forzamos el overfitting
N = 1500
x_train, y_train = x_train_full[:N], y_train_full[:N]
# validación aparte, con la que vigilaremos la generalización
x_val, y_val = x_train_full[N:N+2000], y_train_full[N:N+2000]
print('train:', x_train.shape, ' val:', x_val.shape, ' test:', x_test.shape)

In [ ]:
EPOCHS = 120
BATCH = 64

def train_and_eval(model, callbacks=None, label=''):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    h = model.fit(x_train, y_train,
                  validation_data=(x_val, y_val),
                  epochs=EPOCHS, batch_size=BATCH,
                  callbacks=callbacks or [], verbose=0)
    test_acc = model.evaluate(x_test, y_test, verbose=0)[1]
    print(f'{label:<28} accuracy en TEST = {test_acc:.4f}')
    return h.history, test_acc

## 2 · Sin regularización — el overfitting en directo

Una red grande, sin ninguna defensa. Veremos que la pérdida de **entrenamiento** baja hasta casi cero mientras la de **validación** se estanca y empieza a subir: eso es memorizar.

In [ ]:
def make_base():
    return keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(300, activation='relu'),
        layers.Dense(300, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])

hist_over, acc_over = train_and_eval(make_base(), label='Sin regularización')

In [ ]:
def plot_curves(history, title):
    plt.figure(figsize=(7, 4.5))
    plt.plot(history['loss'], label='train loss')
    plt.plot(history['val_loss'], label='val loss')
    plt.title(title); plt.xlabel('época'); plt.ylabel('loss')
    plt.legend(); plt.grid(alpha=0.3); plt.show()

plot_curves(hist_over, 'Sin regularización — la val_loss se dispara (overfitting)')

**Para observar:** la `train loss` cae hacia 0 (la red memoriza las 1500 imágenes), pero la `val loss` toca un mínimo y luego **sube**. Ese punto de subida es justo donde el early stopping querría parar.

## 3 · Con dropout y regularización L2

Mismo tamaño de red, pero ahora añadimos `Dropout(0.4)` entre capas y un `kernel_regularizer=l2(1e-4)`. La red ya no puede depender de neuronas concretas ni usar pesos enormes.

In [ ]:
def make_regularized():
    reg = regularizers.l2(1e-4)
    return keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(300, activation='relu', kernel_regularizer=reg),
        layers.Dropout(0.4),
        layers.Dense(300, activation='relu', kernel_regularizer=reg),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax'),
    ])

hist_reg, acc_reg = train_and_eval(make_regularized(), label='Dropout + L2')
plot_curves(hist_reg, 'Con dropout + L2 — las curvas ya no se separan tanto')

## 4 · Añadimos early stopping

Con `EarlyStopping` paramos automáticamente cuando la `val_loss` deja de mejorar y **recuperamos el mejor punto** (`restore_best_weights=True`). Así nunca nos pasamos de la raya.

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True)

hist_es, acc_es = train_and_eval(make_regularized(), callbacks=[early],
                                 label='Dropout + L2 + EarlyStopping')
print('Épocas realmente entrenadas:', len(hist_es['loss']))
plot_curves(hist_es, 'Con early stopping — para en el mejor momento')

## 5 · Comparación

Ponemos las tres versiones una al lado de otra. La accuracy de **test** es la medida honesta de generalización.

In [ ]:
print('Accuracy en TEST (generalización):')
print(f'  Sin regularización          : {acc_over:.4f}')
print(f'  Dropout + L2                : {acc_reg:.4f}')
print(f'  Dropout + L2 + EarlyStopping: {acc_es:.4f}')

plt.figure(figsize=(7, 4.5))
plt.plot(hist_over['val_loss'], label='sin regularización')
plt.plot(hist_reg['val_loss'], label='dropout + L2')
plt.plot(hist_es['val_loss'], label='+ early stopping')
plt.title('Pérdida de validación — las tres versiones')
plt.xlabel('época'); plt.ylabel('val_loss'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Para observar:** las versiones regularizadas suelen dar **mejor accuracy en test** que la red sin defensas, aunque su `train loss` sea peor. Recuerda: lo que importa no es acertar en entrenamiento, sino generalizar.

## 6 · Tus retos

1. **Más datos.** Sube `N` de 1500 a 15000 y reentrena la red sin regularización. ¿Se reduce el overfitting? (Más datos es el mejor regularizador.)
2. **Dosis de dropout.** Prueba `Dropout(0.2)` y `Dropout(0.6)`. ¿Demasiado dropout puede provocar underfitting?
3. **La fuerza de L2.** Sube el `l2` a `1e-2`. ¿Qué le pasa a la `train loss`? ¿Y a la de validación?

Cuando termines, vuelve a la [práctica interactiva](../../practica-t8.html) y comprueba que la intuición del slider de grado coincide con lo que has medido aquí.